# Modelisation : premiere comparaison

Comparer trois classifieurs sur les memes donnees : regression logistique, Random Forest et XGBoost.
Le train de 1600 lignes est divise en 1280 lignes d'apprentissage et 320 de validation.
Cette premiere comparaison est simple ; la selection finale se fait dans le notebook 03 par validation croisee.
Aucun modele n'est sauvegarde ici. Prototype pedagogique, sans usage clinique.


## 1. Imports

Les classifieurs partagent fit, predict et predict_proba. train_model construit un pipeline neuf
pour chacun ; evaluate_model calcule les metriques sans refaire fit.
Les imports depuis src permettent de reutiliser le meme code sans le recopier.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.train import train_model
from src.evaluate import evaluate_model


## 2. Split et encodage de la cible

On reprend le split 80/20. Le test de 400 lignes reste a part.
Le train est ensuite divise pour obtenir une validation initiale independante de fit.
OrdinalEncoder encode uniquement y pour XGBoost. categories_[0] donne le sens des codes :
l ordre est fixe explicitement : Low=0, Medium=1, High=2. Les classifieurs restent des classifieurs nominaux.
OneHotEncoder continue de traiter les textes de X dans le preprocessing.


In [2]:
df = pd.read_csv(PROJECT_ROOT / "data/raw/student_mental_health.csv")

df = df.drop_duplicates().dropna(subset=["Mental_Health_Risk"]).copy()

X = df.drop(columns=["Student_ID", "Mental_Health_Risk"])
y = df["Mental_Health_Risk"]

# split
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

X_fit, X_validation, y_fit_text, y_validation_text = train_test_split(
    X_train, 
    y_train, 
    test_size=0.25, 
    random_state=42, 
    stratify=y_train
)


In [10]:
target_encoder = OrdinalEncoder(categories=[["Low", "Medium", "High"]])

y_fit = target_encoder.fit_transform(y_fit_text.to_frame()).astype(int)
y_validation = target_encoder.transform(y_validation_text.to_frame()).astype(int)
class_names = target_encoder.categories_[0]

display(pd.DataFrame({"Code": range(len(class_names)), "Classe": class_names}))


,Code,Classe
0,0,Low
1,1,Medium
2,2,High


## 3. Trois configurations de depart

Regression logistique : reference lineaire, souvent utile quand le signal est simple.
Random Forest : vote de plusieurs arbres, capables de memoriser des details.
XGBoost : arbres successifs avec une contribution controlee par learning_rate.
Ces parametres sont des points de depart, pas des valeurs declarees optimales.


In [11]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, 
        random_state=42
        ),
    "Random Forest": RandomForestClassifier(
        random_state=42
        ),
    "XGBoost": XGBClassifier(
        n_estimators=200, 
        max_depth=3, 
        learning_rate=0.05,
        subsample=0.8, 
        colsample_bytree=0.8,
        objective="multi:softprob", 
        eval_metric="mlogloss",
        tree_method="hist", 
        random_state=42, 
        n_jobs=2,
    ),
}


## 4. Entrainer et evaluer

Pour chaque modele, fit apprend le preprocessing et le classifieur sur X_fit uniquement.
La validation utilise transform/predict avec les regles apprises. Le dictionnaire trained_models
conserve les objets en memoire et comparison contient une ligne de scores par modele.
Precision, rappel et F1 macro donnent le meme poids a chaque classe.


In [12]:
trained_models = {}
evaluations = {}
comparison_rows = []

for name, model in models.items():
    print("in train :", name)

    trained_pipeline = train_model(model, X_fit, y_fit, use_outliers=True)
    trained_models[name] = trained_pipeline

    result = evaluate_model(trained_pipeline, X_validation, y_validation, class_names)
    evaluations[name] = result

    row = {"Modele": name}
    row.update(result["metrics"])
    comparison_rows.append(row)

comparison = pd.DataFrame(comparison_rows).set_index("Modele")
comparison = comparison.sort_values("F1 macro", ascending=False)
display(comparison.round(4))


in train : Logistic Regression


c:\Users\errak\Desktop\Jury Blanc\.venv\Lib\site-packages\sklearn\utils\validation.py:1368: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\errak\Desktop\Jury Blanc\.venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


in train : Random Forest
in train : XGBoost


,Accuracy,Precision macro,Recall macro,F1 macro,ROC AUC macro OvR
Modele,,,,,
Logistic Regression,0.745,0.7509,0.7440,0.7457,0.9046
Random Forest,0.700,0.7011,0.6991,0.6999,0.8568
XGBoost,0.690,0.6962,0.6894,0.6920,0.8676
